# 共同随机趋势与平稳价差
先修：AR(1)、OLS、单位根和协整的定义。先区分确定性算例与随机模型，再估计长期关系、误差修正和 OU 时间尺度，最后重复模拟看不确定性。

原来的 $0.55^t$ 是没有创新的确定性衰减，只适合核对代数，不是平稳 AR(1) 样本。下面先保留这种形状，再生成有独立正态创新、正确平稳初值的价差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen
rng = np.random.default_rng(2026)
n, phi, noise = 300, .8, .5
a, beta = 1.5, 2.
deterministic = .55 ** np.arange(30)
x = np.cumsum(rng.normal(0, 1, n))
z = np.empty(n)
z[0] = rng.normal(0, noise/np.sqrt(1-phi**2))
for t in range(1, n):
    z[t] = phi*z[t-1] + rng.normal(0, noise)
y = a + beta*x + z
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(deterministic); axes[0].set_title('deterministic decay')
axes[1].plot(z); axes[1].set_title('stationary AR(1) realization'); plt.show()

初值方差 $\sigma_\eta^2/(1-\phi^2)$ 使整个残差过程平稳。随机游走 $x$ 的方差随时间增长，而 $y-1.5-2x=z$ 的方差恒定。先预测：OLS 是否每次都恰好恢复斜率 2？

In [ ]:
design = np.column_stack([np.ones(n), x])
estimated = np.linalg.lstsq(design, y, rcond=None)[0]
residual = y-design@estimated
print('截距、斜率：', estimated)
# 事先选定含常数、最多 1 个差分滞后，关闭自动选阶。
eg_stat, eg_p, eg_critical = coint(y, x, trend='c', maxlag=1, autolag=None)
print('Engle-Granger 统计量、p 值：', eg_stat, eg_p)
print('真实 z 的 ADF（另一检验问题）：', adfuller(z, regression='c', maxlag=1, autolag=None)[:2])
fig, ax = plt.subplots(); ax.plot(z, label='true spread'); ax.plot(residual, label='estimated spread', alpha=.7)
ax.legend(); plt.show()

估计残差的单位根检验要使用协整检验的临界值，不能将普通 ADF 的 p 值直接当作 Engle–Granger 的 p 值。两格结果都有有限样本不确定性。多变量情形的 Johansen 检验将长期关系数目变成秩问题；下格仅作选读比较。

In [ ]:
johansen = coint_johansen(np.column_stack([y, x]), det_order=0, k_ar_diff=1)
print('Johansen trace：', johansen.lr1, '95% 临界值：', johansen.cvt[:, 1])
ecm_design = np.column_stack([np.ones(n-1), residual[:-1], np.diff(x)])
ecm = np.linalg.lstsq(ecm_design, np.diff(y), rcond=None)[0]
ar = np.linalg.lstsq(np.column_stack([np.ones(n-1), residual[:-1]]), residual[1:], rcond=None)[0]
estimated_phi = ar[1]
print('ECM 调整系数（理论 phi-1）：', ecm[1], phi-1)
if 0 < estimated_phi < 1:
    kappa = -np.log(estimated_phi)  # 采样间隔 1
    innovations = residual[1:] - ar[0] - estimated_phi*residual[:-1]
    sigma_ou = np.sqrt(innovations.var(ddof=2)*2*kappa/(1-estimated_phi**2))
    print('OU kappa、扩散率、半衰期：', kappa, sigma_ou, np.log(2)/kappa)
else:
    print('此样本估计不满足正均值回归 OU 的映射条件。')

## 重复实验
下面把已经展示的模拟与回归放入小函数，只为重复计算。比较样本量、持续性和噪声；不要只挑一条看起来理想的路径。

In [ ]:
def one_sample(count, persistence, scale, generator):
    xx = np.cumsum(generator.normal(size=count))
    zz = np.empty(count)
    zz[0] = generator.normal(0, scale/np.sqrt(1-persistence**2))
    for t in range(1, count):
        zz[t] = persistence*zz[t-1] + generator.normal(0, scale)
    yy = 1.5 + 2*xx + zz
    slope = np.linalg.lstsq(np.column_stack([np.ones(count), xx]), yy, rcond=None)[0][1]
    pvalue = coint(yy, xx, trend='c', maxlag=1, autolag=None)[1]
    return slope, pvalue

for count, persistence, scale in [(80, .8, .5), (300, .8, .5), (300, .97, .5), (300, .8, 2.)]:
    draws = np.array([one_sample(count, persistence, scale, rng) for _ in range(60)])
    print((count, persistence, scale), '斜率 10/50/90 分位', np.quantile(draws[:, 0], [.1, .5, .9]),
          '5% 水平拒绝频率', np.mean(draws[:, 1]<.05))

### 半衰期与一个透明 DF 回归
下格用不含常数、零差分滞后的同一规格比较回归统计量；估计残差的正式协整显著性仍用前面的专用检验。

In [ ]:
phis = np.linspace(.01, .99, 100)
fig, ax = plt.subplots()
ax.plot(phis, -np.log(2)/np.log(phis))
ax.set(xlabel='phi (valid OU range: 0 < phi < 1)', ylabel='half-life')
plt.show()
lag = residual[:-1]
dy = np.diff(residual)
gamma = (lag @ dy) / (lag @ lag)
error = dy-gamma*lag
se = np.sqrt((error @ error)/(len(dy)-1)/(lag @ lag))
print('透明 DF / 相同规格 adfuller：', gamma/se,
      adfuller(residual, maxlag=0, regression='n', autolag=None)[0])

## 练习与反馈
把持续性改为 0.97，说明为什么短样本难以辨认均值回归。拒绝频率是此模拟设计下的功效估计，不是“协整为真的概率”；60 次重复本身也有 Monte Carlo 误差。

当 $\phi$ 接近 1 时，冲击消退慢，短期路径类似随机游走。半衰期描述条件均值偏离减半，不是随机首次命中均值的期望时间。公开宏观快照与原确定性算例仍可在研究材料中用于数据和代数核对；它们不被当作可交易市场证据。